In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from tvDatafeed import TvDatafeed, Interval



In [113]:


# Login no TradingView
tv = TvDatafeed()

ticker = 'WIN1!'
exchange = 'BMFBOVESPA'


df = tv.get_hist(
    symbol=ticker,
    exchange=exchange,
    interval=Interval.in_daily,
    n_bars=10000
)
df = df[df.index.year >=2000].dropna()
df.index = pd.to_datetime(df.index).normalize().date
df.drop(columns='symbol',inplace=True)
df.index = pd.to_datetime(df.index).normalize()
df.dropna(inplace=True)
df['ret'] = df['close'].pct_change()
df.tail()




,open,high,low,close,volume,ret
2025-12-15,161575.0,163300.0,161345.0,162808.0,18892438.0,0.011368
2025-12-16,161250.0,161350.0,157325.0,158219.0,18346181.0,-0.028187
2025-12-17,161500.0,161690.0,159610.0,160617.0,17583079.0,0.015156
2025-12-18,160660.0,161835.0,159845.0,161296.0,17220599.0,0.004227
2025-12-19,161200.0,162725.0,160990.0,161574.0,14265906.0,0.001724


In [114]:
n = 3
df["neg"] = df["ret"] < 0
neg_streak = []
count = 0

for is_neg in df["neg"]:
    if is_neg:
        count += 1
    else:
        count = 0
    neg_streak.append(count)

df["neg_streak"] = neg_streak
df["entry_signal"] = df["neg_streak"] == n

position = 0
positions = []

for ret, entry in zip(df["ret"], df["entry_signal"]):
    if position == 0 and entry:
        position = 1
    elif position == 1 and ret > 0:
        position = 0
    positions.append(position)

df["position"] = positions
df["strategy_ret"] = df["position"].shift(1) * df["ret"]


df["strategy"] = df["strategy_ret"].cumsum()
df["buy_hold"] = df["ret"].cumsum()
df.tail()

,open,high,low,close,volume,ret,neg,neg_streak,entry_signal,position,strategy_ret,strategy,buy_hold
2025-12-15,161575.0,163300.0,161345.0,162808.0,18892438.0,0.011368,False,0,False,0,0.0,0.871052,2.451646
2025-12-16,161250.0,161350.0,157325.0,158219.0,18346181.0,-0.028187,True,1,False,0,-0.0,0.871052,2.423460
2025-12-17,161500.0,161690.0,159610.0,160617.0,17583079.0,0.015156,False,0,False,0,0.0,0.871052,2.438616
2025-12-18,160660.0,161835.0,159845.0,161296.0,17220599.0,0.004227,False,0,False,0,0.0,0.871052,2.442843
2025-12-19,161200.0,162725.0,160990.0,161574.0,14265906.0,0.001724,False,0,False,0,0.0,0.871052,2.444567


In [ ]:
# =========================
# PLOT
# =========================
fig = make_subplots(
    rows=1,
    cols=1,
    shared_xaxes=True,
    specs=[[{"secondary_y": True}]]
)

# Buy & Hold
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["buy_hold"] * 100,
        name="Buy & Hold",
        line=dict(width=2)
    ),
    secondary_y=False
)

# Strategy
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df["strategy"] * 100,
        name="Strategy",
        line=dict(width=2)
    ),
    secondary_y=False
)


mask = df["position"] == 1

fig.add_trace(
    go.Scatter(
        x=df.index[mask],
        y=df.loc[mask, "buy_hold"]*100,
        mode="markers",
        name="Buy Signal",
        marker=dict(
            symbol="triangle-up",
            size=10
        )
    ),
    secondary_y=False
)

# Layout
fig.update_layout(
    title=f"{ticker} | compra após {n} dias de queda",
    xaxis_title="Date",
    yaxis_title="Cumulative Return (%)",
    yaxis2_title="Position",
    legend=dict(x=0.01, y=0.99),
    template="plotly_white",
    height=600
)

# Ajuste do eixo secundário
fig.update_yaxes(range=[-0.05, 1.05], secondary_y=True)

fig.show()



In [116]:
total_return_bh = df["buy_hold"].iloc[-1]
total_return_strategy = df["strategy"].iloc[-1]

vol_strategy = df["strategy_ret"].std() * np.sqrt(252)
vol_bh = df["ret"].std() * np.sqrt(252)

sharpe_strategy = (
    df["strategy_ret"].mean() / df["strategy_ret"].std()
) * np.sqrt(252)

sharpe_bh = (
    df["ret"].mean() / df["ret"].std()
) * np.sqrt(252)

print("=== RESULTADOS ===")
print(f"Buy & Hold Return: {total_return_bh:.2%}")
print(f"Strategy Return:   {total_return_strategy:.2%}")
print()
print(f"Buy & Hold Vol: {vol_bh:.2%}")
print(f"Strategy Vol:   {vol_strategy:.2%}")
print()
print(f"Buy & Hold Sharpe: {sharpe_bh:.2f}")
print(f"Strategy Sharpe:   {sharpe_strategy:.2f}")


=== RESULTADOS ===
Buy & Hold Return: 244.46%
Strategy Return:   87.11%

Buy & Hold Vol: 26.62%
Strategy Vol:   9.30%

Buy & Hold Sharpe: 0.45
Strategy Sharpe:   0.46
